# U.S. Airline Arrival Delay Prediction

**Course:** CS-GY 6513-C Big Data  
**Project:** U.S. Airline Flight Delay Analysis  
**Goal:** Build a simple Spark-based machine learning model to estimate whether a flight will arrive late.

This notebook uses the cleaned airline dataset created from the data engineering pipeline and focuses on the predictive analysis part of the project.

## 1. Import Libraries and Start Spark

We use PySpark because the cleaned airline dataset contains millions of records. Spark allows us to process and model the data in a distributed and scalable way.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, when
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

import pandas as pd
import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("AirlineDelayPrediction") \
    .getOrCreate()

print("Spark session started successfully.")

## 2. Load the Cleaned Dataset

The cleaned dataset was exported from the data cleaning notebook as a Spark output folder named `cleaned_airline_data`. Spark can directly read this folder.

In [ ]:
df = spark.read.csv(
    "cleaned_airline_data",
    header=True,
    inferSchema=True
)

print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()

## 3. Preview the Data

This step confirms that the cleaned dataset loaded correctly.

In [ ]:
df.show(5, truncate=False)

## 4. Check Target Variable

The target variable for prediction is `is_arr_delayed`.

- `0` means the flight was not delayed on arrival.
- `1` means the flight was delayed on arrival.

In [ ]:
target_dist = df.groupBy("is_arr_delayed").count().orderBy("is_arr_delayed")
target_dist.show()

target_pd = target_dist.toPandas()

plt.figure(figsize=(6, 4))
plt.bar(target_pd["is_arr_delayed"].astype(str), target_pd["count"])
plt.xlabel("Arrival Delay Label")
plt.ylabel("Number of Flights")
plt.title("Target Distribution: Arrival Delay")
plt.show()

## 5. Exploratory Analysis: Delay Rate by Airline

This visualization supports the project objective of comparing airline delay performance.

In [ ]:
airline_delay = df.groupBy("Reporting_Airline") \
    .agg(avg("is_arr_delayed").alias("delay_rate")) \
    .orderBy(col("delay_rate").desc())

airline_delay.show(20)

airline_pd = airline_delay.toPandas()

plt.figure(figsize=(10, 5))
plt.bar(airline_pd["Reporting_Airline"], airline_pd["delay_rate"])
plt.xlabel("Airline")
plt.ylabel("Average Arrival Delay Rate")
plt.title("Arrival Delay Rate by Airline")
plt.xticks(rotation=45)
plt.show()

## 6. Exploratory Analysis: Delay Rate by Month

This visualization helps identify temporal delay patterns across the year.

In [ ]:
month_delay = df.groupBy("Month") \
    .agg(avg("is_arr_delayed").alias("delay_rate")) \
    .orderBy("Month")

month_delay.show()

month_pd = month_delay.toPandas()

plt.figure(figsize=(8, 5))
plt.plot(month_pd["Month"], month_pd["delay_rate"], marker="o")
plt.xlabel("Month")
plt.ylabel("Average Arrival Delay Rate")
plt.title("Arrival Delay Rate by Month")
plt.xticks(range(1, 13))
plt.show()

## 7. Exploratory Analysis: Delay Rate by Day of Week

This step examines whether arrival delays are more common on certain days.

In [ ]:
dow_delay = df.groupBy("DayOfWeek") \
    .agg(avg("is_arr_delayed").alias("delay_rate")) \
    .orderBy("DayOfWeek")

dow_delay.show()

dow_pd = dow_delay.toPandas()

plt.figure(figsize=(8, 5))
plt.bar(dow_pd["DayOfWeek"], dow_pd["delay_rate"])
plt.xlabel("Day of Week")
plt.ylabel("Average Arrival Delay Rate")
plt.title("Arrival Delay Rate by Day of Week")
plt.show()

## 8. Exploratory Analysis: Delay Rate by Season

This supports the project objective of analyzing seasonal flight delay trends.

In [ ]:
season_delay = df.groupBy("season") \
    .agg(avg("is_arr_delayed").alias("delay_rate")) \
    .orderBy(col("delay_rate").desc())

season_delay.show()

season_pd = season_delay.toPandas()

plt.figure(figsize=(7, 5))
plt.bar(season_pd["season"], season_pd["delay_rate"])
plt.xlabel("Season")
plt.ylabel("Average Arrival Delay Rate")
plt.title("Arrival Delay Rate by Season")
plt.show()

## 9. Exploratory Analysis: Top Origin Airports by Delay Rate

This supports the airport congestion analysis objective by identifying airports with higher delay rates.

In [ ]:
airport_delay = df.groupBy("Origin") \
    .agg(
        count("*").alias("flight_count"),
        avg("is_arr_delayed").alias("delay_rate")
    ) \
    .filter(col("flight_count") > 10000) \
    .orderBy(col("delay_rate").desc())

airport_delay.show(15)

airport_pd = airport_delay.limit(15).toPandas()

plt.figure(figsize=(10, 5))
plt.bar(airport_pd["Origin"], airport_pd["delay_rate"])
plt.xlabel("Origin Airport")
plt.ylabel("Average Arrival Delay Rate")
plt.title("Top 15 Origin Airports by Arrival Delay Rate")
plt.xticks(rotation=45)
plt.show()

## 10. Feature Selection for Prediction

To avoid data leakage, we only use features that are known before the flight departs.

We do **not** use columns such as `ArrDelay`, `ArrTime`, `DepDelay`, `DepTime`, or delay-cause columns because they reveal information that would only be known after the flight has already happened.

In [ ]:
model_df = df.select(
    "Year",
    "Quarter",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSElapsedTime",
    "Distance",
    "dep_hour",
    "season",
    "is_arr_delayed"
).dropna()

print("Rows after selecting modeling columns:", model_df.count())
model_df.show(5)

## 11. Define Categorical and Numerical Features

Categorical features are converted into numeric indexes using `StringIndexer`. Numerical features are passed directly into the feature vector.

In [ ]:
categorical_cols = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "season"
]

numeric_cols = [
    "Year",
    "Quarter",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "CRSDepTime",
    "CRSElapsedTime",
    "Distance",
    "dep_hour"
]

label_col = "is_arr_delayed"

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

indexed_cols = [c + "_idx" for c in categorical_cols]

feature_cols = indexed_cols + numeric_cols

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

## 12. Train-Test Split

The dataset is split into training and testing sets. The training set is used to fit the model, and the testing set is used to evaluate performance on unseen data.

In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

## 13. Evaluation Function

We evaluate the models using:

- Accuracy
- F1 Score
- Precision
- Recall
- AUC

AUC is useful because the project goal is to estimate the probability of flight delays.

In [ ]:
def evaluate_model(predictions, model_name):
    accuracy = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction",
        metricName="accuracy"
    ).evaluate(predictions)

    f1 = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction",
        metricName="f1"
    ).evaluate(predictions)

    precision = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction",
        metricName="weightedPrecision"
    ).evaluate(predictions)

    recall = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction",
        metricName="weightedRecall"
    ).evaluate(predictions)

    auc = BinaryClassificationEvaluator(
        labelCol=label_col,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    ).evaluate(predictions)

    print(f"===== {model_name} Results =====")
    print("Accuracy:", accuracy)
    print("F1 Score:", f1)
    print("Precision:", precision)
    print("Recall:", recall)
    print("AUC:", auc)

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "F1 Score": f1,
        "Precision": precision,
        "Recall": recall,
        "AUC": auc
    }

## 14. Model 1: Logistic Regression

Logistic Regression is used as the main model because the project proposal asks for a simple machine learning model that estimates the probability of flight delays.

In [ ]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    maxIter=10
)

lr_pipeline = Pipeline(stages=indexers + [assembler, lr])

lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

lr_results = evaluate_model(lr_predictions, "Logistic Regression")

## 15. Model 2: Random Forest Classifier

Random Forest is added as a comparison model. It can capture non-linear relationships between features and arrival delays.

In [ ]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    numTrees=30,
    maxDepth=8,
    seed=42
)

rf_pipeline = Pipeline(stages=indexers + [assembler, rf])

rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)

rf_results = evaluate_model(rf_predictions, "Random Forest")

## 16. Compare Model Performance

This table compares the two models using the selected evaluation metrics.

In [ ]:
results = [lr_results, rf_results]
results_pd = pd.DataFrame(results)
results_pd

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_pd["Model"], results_pd["Accuracy"])
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison")
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(results_pd["Model"], results_pd["F1 Score"])
plt.xlabel("Model")
plt.ylabel("F1 Score")
plt.title("Model F1 Score Comparison")
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(results_pd["Model"], results_pd["AUC"])
plt.xlabel("Model")
plt.ylabel("AUC")
plt.title("Model AUC Comparison")
plt.show()

## 17. Confusion Matrix

The confusion matrix shows how many flights were correctly and incorrectly classified as delayed or not delayed.

In [ ]:
# Choose the better model after comparing the metrics.
# By default, this uses Random Forest predictions.
best_predictions = rf_predictions

conf_matrix = best_predictions.groupBy(label_col, "prediction").count()
conf_matrix.show()

conf_pd = conf_matrix.toPandas()

pivot_conf = conf_pd.pivot(
    index=label_col,
    columns="prediction",
    values="count"
).fillna(0)

pivot_conf

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(pivot_conf, aspect="auto")
plt.colorbar()
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix")

plt.xticks(range(len(pivot_conf.columns)), pivot_conf.columns)
plt.yticks(range(len(pivot_conf.index)), pivot_conf.index)

for i in range(len(pivot_conf.index)):
    for j in range(len(pivot_conf.columns)):
        plt.text(j, i, int(pivot_conf.iloc[i, j]), ha="center", va="center")

plt.show()

## 18. Feature Importance from Random Forest

Feature importance helps identify which variables were most useful for predicting arrival delays.

In [ ]:
rf_stage = rf_model.stages[-1]

feature_importance_pd = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_stage.featureImportances.toArray()
}).sort_values(by="Importance", ascending=False)

feature_importance_pd

In [ ]:
top_features = feature_importance_pd.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Feature Importances - Random Forest")
plt.gca().invert_yaxis()
plt.show()

## 19. Sample Predictions

This displays sample predictions, including the predicted label and probability.

In [ ]:
best_predictions.select(
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Month",
    "DayOfWeek",
    "CRSDepTime",
    "Distance",
    "is_arr_delayed",
    "prediction",
    "probability"
).show(20, truncate=False)

## 20. Final Summary

This notebook developed a Spark-based predictive model for airline arrival delays. The model uses historical schedule-based flight features such as airline, origin airport, destination airport, scheduled departure time, distance, month, day of week, departure hour, and season.

The main predictive model is Logistic Regression because it is simple and provides probability estimates. Random Forest is included as a comparison model to evaluate whether a tree-based method improves performance.

Columns that would cause data leakage, such as actual arrival delay and delay-cause variables, were excluded from the model. This makes the prediction setup more realistic because the model only uses information that would be available before the flight occurs.

In [ ]:
print("""
Final Summary:
A Spark ML pipeline was developed to predict whether a flight will arrive late.

Target variable:
- is_arr_delayed

Main model:
- Logistic Regression

Comparison model:
- Random Forest

The models used schedule-based historical features and avoided leakage columns such as
actual arrival delay, actual departure delay, and delay-cause variables.
""")